# പാഠം 08 - മള്‍ട്ടി-എജന്റ് ഡിസൈൻ പാറ്റേൺ


## സെറ്റപ്പ്


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾക്ക് എന്തുകൊണ്ട്?

യാത്രാ പദ്ധതിയിടൽ പോലുള്ള യാഥാർഥ്യ ജോലികൾ പലവിധ വൈശേഷ്യങ്ങളും ആവശ്യപ്പെടുന്നു — ലജിസ്റ്റിക്സ്, പ്രാദേശിക ജ്ഞാനം, ബഡ്ജറ്റിൽ തുടങ്ങിയവ. എല്ലാം കൈകാര്യം ചെയ്യാൻ ശ്രമിക്കുന്ന ഏക ഏജന്റ് വളരെ വേഗം നിയന്ത്രിക്കാൻ കഴിയാത്തതാകുന്നു.

മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ ഇതു **വിശേഷതാഭിപ്രായം** മുഖേന പരിഹരിക്കുന്നു: ഓരോ ഏജന്റും ഒരു പ്രത്യേക മേഖലയിൽ ശ്രദ്ധ കേന്ദ്രീകരിക്കുന്നു, പൊതു നൈപുണ്യമുള്ളവരെക്കാൾ ഉയർന്ന നിലവാരമുള്ള ഫലങ്ങൾ നൽകുന്നു. കൂടാതെ, അവ **വ്യാപ്തി** മെച്ചപ്പെടുത്തുന്നു — പുതിയ ഏജന്റുകളെ (ഉദാ: ഒരു വിമാന വിദഗ്ധൻ, ഒരു റസ്റ്റോറന്റ് വിമർശകൻ) പുതിയതായി ചേർക്കാൻ കഴിഞ്ഞാലും നിലവിലുള്ള പ്രവർത്തനക്രമം മാറ്റേണ്ടതില്ല. ഏജന്റുകൾ ഒരു ഘടനയുള്ള പൈപ്പ്ലൈനിലൂടെ ചേർന്നു പ്രവർത്തിക്കുന്നു, ഒന്ന് മറ്റൊന്നിലേക്ക് പുറപ്പെട്ടിരിക്കുന്ന പരിസര ഘടകങ്ങൾ കൈമാറുന്നു.


## പ്രത്യേകജ്ഞാന ഏജന്റുമാർ സൃഷ്ടിക്കൽ


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ഒരു ക്രമബന്ധിത ജോലിനിര എടുക്കല്‍

`WorkflowBuilder` നിങ്ങളെ ഏജന്റുകളെ ഒരു ദിശാബദ്ധമായ ഗ്രാഫായി ബന്ധിപ്പിക്കാന്‍ അനുവദിക്കുന്നു. ഇവിടെ നാം ഒരു ലളിതമായ രണ്ട്-പടി പൈപ്പ്ലൈന്‍ സൃഷ്ടിക്കുന്നു: **TravelPlanner** യാത്രാമാപ്പ് രൂപകല്‍പന ചെയ്യുന്നു, പിന്നെ **TravelConcierge** അതിനെ പരിശോധിച്ച് മെച്ചപ്പെടുത്തുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ഒഴിവ് വർക്ക്‌ഫ്ലോയിൽ കൂടുതൽ ഏജന്റുകൾ ചേർക്കൽ

മൾട്ടി-ഏജന്റ് പാറ്റേണിന്റെ ഏറ്റവും വലിയ ഗുണങ്ങളിൽ ഒന്ന് അതിനെ എളുപ്പത്തിൽ വിപുലീകരിക്കാൻ കഴിയുന്നതാണ്. താഴെ, യാത്രക്കാരന്റെ ബജറ്റുമായി താരതമ്യം ചെയ്ത് പദ്ധതിയെ പരിശോധിക്കുന്ന, ചില ഐറ്റങ്ങൾ ചെലവു പരിധിക്ക് മുകളിൽ പോകാമെന്ന് സൂചിപ്പിക്കുന്ന, പണസംരക്ഷണങ്ങൾ അനുകൂലിക്കുന്ന ബഡ്ജറ്റ്_റിവ്യൂവർ എന്ന ഏജന്റിനെ ചേർക്കുന്നു. വർക്ക്‌ഫ്ലോ ഇപ്പോള്‍ മൂന്ന് ഏജന്റുകളെ തുടർച്ചയായി പ്രവർത്തിപ്പിക്കുന്നു:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## സംക്ഷിപ്തം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. **നിരവധി പ്രത്യേക ഏജൻറുകൾ സൃഷ്ടിക്കുക** — ഓരോന്നിന്റെയും ശ്രദ്ധ കേന്ദ്രീകരിച്ചുള്ള പങ്ക് (പതിവ്, കോൺസിയർജ്, ബഡ്ജറ്റ് അവലോകനം).
2. **`WorkflowBuilder` ഉം `add_edge` ഉം ഉപയോഗിച്ച് ഏജൻറുകളെ അനുക്രമമായ പ്രവൃത്തി പ്രവാഹത്തിൽ ബന്ധിപ്പിക്കുക**.
3. **മൾട്ടി-ഏജന്റ് പൈപ്പ്‌ലൈൻ നിന്നുള്ള ഔട്ട്പുട്ട് സ്ട്രീം ചെയ്യുക**, ഏജന്റ് ആരാണെന്ന് പിന്തുടരാം.
4. **നിലവിലുള്ള ഏജൻറുകൾ മാറ്റമില്ലാതെ ചൈനിൽ പുതിയ ഏജൻറുകൾ ചേർത്ത് പ്രവൃത്തി പ്രവാഹം വിപുലപ്പെടുത്തുക**.

മൾട്ടി-ഏജന്റ് ഡിസൈൻ പാറ്റേൺ ഓരോ ഏജന്റിനും ലളിതമായ നില കൊടുക്കുന്നതോടൊപ്പം, ഒരേയൊരു ഏജന്റ് മാത്രം സാധ്യമായ പോലെ സമൃദ്ധവും കൂടുതൽ പൂർണമായ അവലോകനഫലിതങ്ങളും ഉത്പാദിപ്പിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
